# RAG-Powered Search Engine — Q&A with Retrieval-Augmented Generation
### Final Capstone Project · IITG · AI Application Development

**Author:** _Prasad Thipparthi_
**Project option:** #2 — RAG-Powered Search Engine
**Stack:** LangChain · Azure OpenAI (`gpt-5-chat`) · local MiniLM embeddings · FAISS · Streamlit

---
This notebook is the **end-to-end** implementation of a Retrieval-Augmented
Generation (RAG) question-answering system. It walks through the problem
statement, every step of the pipeline (with explanations), a testing and
evaluation section, and the final results.

> The core logic lives in [`src/rag_pipeline.py`](src/rag_pipeline.py) so that
> this notebook and the deployment app (`app.py`) run **exactly the same code**.


## 1. Problem Statement

Large Language Models (LLMs) are powerful but have two well-known limitations:

1. **Knowledge cut-off & private data** — an LLM only "knows" what was in its
   training data. It cannot answer questions about your private PDFs, internal
   wikis, or documents published after its training cut-off.
2. **Hallucination** — when an LLM does not know something, it often produces a
   confident but incorrect answer.

**Goal:** Build a search engine that answers natural-language questions about a
*custom document collection*. The system must:

- Retrieve the most relevant passages from the documents for each question.
- Generate an answer **grounded** in those passages.
- **Cite its sources** so the answer is verifiable.
- Say *"I don't know"* when the documents do not contain the answer.

The solution is **Retrieval-Augmented Generation (RAG)**: combine a vector
similarity search over the documents with an LLM that is instructed to answer
only from the retrieved context.


## 2. Setup & Dependencies

Install the dependencies (already pinned in `requirements.txt`). Uncomment the
line below the first time you run the notebook.


In [ ]:
# !pip install -r requirements.txt
import sys, os
print("Python:", sys.version.split()[0])


### Configure your provider

This project uses **Azure OpenAI** for the chat LLM and a **local** embedding
model (so no embeddings deployment or extra API cost is needed). Configuration is
read from a `.env` file:

```
AZURE_OPENAI_ENDPOINT=https://<resource>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=<key>
AZURE_OPENAI_DEPLOYMENT=gpt-5-chat
AZURE_OPENAI_API_VERSION=2025-01-01-preview
EMBEDDINGS_BACKEND=local
```

The pipeline auto-detects the provider: if `AZURE_OPENAI_ENDPOINT` is set it uses
Azure, otherwise it falls back to standard OpenAI (`OPENAI_API_KEY`). The cell
below verifies the configuration.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

if os.getenv("AZURE_OPENAI_ENDPOINT"):
    print("Azure OpenAI configured - deployment:", os.getenv("AZURE_OPENAI_DEPLOYMENT"))
elif os.getenv("OPENAI_API_KEY"):
    print("Standard OpenAI configured.")
else:
    print("WARNING: No LLM provider configured. Set Azure or OpenAI env vars in .env.")

print("Embeddings backend:", os.getenv("EMBEDDINGS_BACKEND", "local"))


## 3. Architecture Overview

The RAG pipeline has two phases.

**Indexing (offline):**
```
documents  ->  load  ->  split into chunks  ->  embed  ->  FAISS vector store
```

**Querying (online):**
```
question -> embed -> retrieve top-k chunks -> build prompt (context+question)
         -> LLM generates grounded answer -> answer + cited sources
```

We implement each step explicitly below using the shared `RAGPipeline` class.


In [ ]:
# Make the src/ package importable from the notebook.
sys.path.append(os.path.abspath("src"))
from rag_pipeline import RAGPipeline, RAGConfig

config = RAGConfig(
    data_dir="data",
    chunk_size=1000,
    chunk_overlap=150,
    top_k=4,
    temperature=0.0,
)
# Note: the chat model (Azure deployment) and embedding backend are resolved
# from environment variables at call time, so the config stays provider-agnostic.
rag = RAGPipeline(config)
config


## 4. Step 1 — Load the Documents

We load every supported file (`.md`, `.txt`, `.pdf`) from the `data/` folder.
The sample corpus contains three documents about RAG, LLM concepts, and vector
databases. Each loaded `Document` carries its text plus `source` metadata used
later for citations.


In [ ]:
docs = rag.load_documents()
print(f"Loaded {len(docs)} document(s):")
for d in docs:
    print(f"  - {d.metadata['source']:25s} ({len(d.page_content)} chars)")


## 5. Step 2 — Split into Chunks

LLMs and embedding models have bounded context windows, and retrieval works best
on focused passages. We split documents into overlapping chunks using
`RecursiveCharacterTextSplitter`, which tries to break on paragraph, then line,
then sentence boundaries.

- **chunk_size = 1000** characters keeps each chunk topically focused.
- **chunk_overlap = 150** characters preserves context that straddles a boundary.


In [ ]:
chunks = rag.split_documents()
print(f"Split {len(docs)} document(s) into {len(chunks)} chunks.")
print("\nExample chunk (truncated):\n")
print(chunks[0].page_content[:400], "...")
print("\nMetadata:", chunks[0].metadata)


## 6. Step 3 — Embed & Build the FAISS Vector Store

Each chunk is converted into a dense vector with a **local** embedding model
(`all-MiniLM-L6-v2`, 384 dimensions) and stored in a **FAISS** index. FAISS
performs fast similarity nearest-neighbour search entirely in memory — perfect
for a prototype. Running embeddings locally means this step is free and offline.

> The first run downloads the embedding model (~90 MB) once.


In [ ]:
vector_store = rag.build_index()
print("FAISS index built.")
print("Number of vectors indexed:", vector_store.index.ntotal)
print("Embedding dimension:", vector_store.index.d)


### (Optional) Persist the index to disk

Saving the index avoids re-embedding the documents on every run.


In [ ]:
path = rag.save_index("faiss_index")
print("Index saved to:", path)
# Reload later with:  rag.load_index("faiss_index")


## 7. Step 4 — Retrieval (inspect what the engine finds)

Before generating an answer, let's see which chunks the retriever returns for a
question. This is the *augmentation* in Retrieval-Augmented Generation.


In [ ]:
retrieved = rag.retrieve_only("Why does RAG reduce hallucination?")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, d in enumerate(retrieved, 1):
    print(f"[{i}] source={d.metadata['source']}")
    print("    ", d.page_content[:160].replace("\n", " "), "...\n")


## 8. Step 5 — Generate a Grounded Answer

The `query()` method runs the full LCEL chain:
`retrieve -> build prompt (context + question) -> LLM -> parse`. The system
prompt instructs the model to answer **only** from the retrieved context and to
abstain otherwise. It returns the answer and the source chunks.


In [ ]:
result = rag.query("What is RAG and why is it useful?")
print("ANSWER:\n", result["answer"])
print("\nSOURCES:", sorted({d.metadata['source'] for d in result['sources']}))


## 9. Testing & Evaluation

We evaluate the system on two axes.

### 9a. Retrieval quality — `hit@k`
For a set of questions with a known *expected source document*, we check whether
that document appears among the top-k retrieved chunks. `hit@k` is the fraction
of questions for which retrieval succeeded.

### 9b. Generation quality — faithfulness & abstention
We confirm the model (1) answers in-corpus questions correctly and (2) correctly
abstains ("I don't know...") on out-of-corpus questions — the key anti-
hallucination behaviour.


In [ ]:
# Evaluation set: question -> document we expect to be retrieved.
eval_set = [
    {"q": "What is a token in an LLM?",                 "expected": "llm_concepts.md"},
    {"q": "What distance metric is common for text embeddings?", "expected": "vector_databases.md"},
    {"q": "What chunk size is a good starting point for RAG?",   "expected": "rag_overview.md"},
    {"q": "What does temperature 0 do during generation?",       "expected": "llm_concepts.md"},
    {"q": "What is FAISS and why use it for prototypes?",        "expected": "vector_databases.md"},
]

hits = 0
for item in eval_set:
    srcs = {d.metadata["source"] for d in rag.retrieve_only(item["q"])}
    hit = item["expected"] in srcs
    hits += hit
    print(f"{'OK ' if hit else 'MISS'} | {item['q'][:50]:50s} -> {sorted(srcs)}")

hit_at_k = hits / len(eval_set)
print(f"\nhit@{config.top_k} = {hit_at_k:.0%}  ({hits}/{len(eval_set)})")


In [ ]:
# Generation quality: in-corpus correctness + out-of-corpus abstention.
in_corpus = "What are the two phases of a RAG pipeline?"
out_corpus = "What was the stock price of Apple on 1 January 2026?"

ans_in = rag.query(in_corpus)["answer"]
ans_out = rag.query(out_corpus)["answer"]

print("IN-CORPUS  Q:", in_corpus)
print("           A:", ans_in, "\n")
print("OUT-CORPUS Q:", out_corpus)
print("           A:", ans_out)

abstained = "i don't know" in ans_out.lower()
print("\nCorrectly abstained on out-of-corpus question:", abstained)


### 9c. Latency sanity check
A quick wall-clock timing of a single query (retrieval + generation).


In [ ]:
import time
t0 = time.time()
_ = rag.query("How does chunk overlap help retrieval?")
print(f"End-to-end query latency: {time.time() - t0:.2f} s")


## 10. Final Results

A short batch of representative questions, with answers and citations.


In [ ]:
questions = [
    "What is RAG and why is it useful?",
    "What is the difference between cosine similarity and Euclidean distance?",
    "Why is temperature 0 preferred for factual question answering?",
    "What metrics evaluate retrieval quality?",
]

for q in questions:
    r = rag.query(q)
    print("Q:", q)
    print("A:", r["answer"])
    print("Sources:", sorted({d.metadata['source'] for d in r['sources']}))
    print("-" * 90)


## 11. Conclusion, Challenges & Future Improvements

**What we built.** A complete RAG search engine: documents are loaded, chunked,
embedded (locally) into a FAISS vector store, and queried through a LangChain
LCEL chain that grounds an Azure OpenAI `gpt-5-chat` answer in retrieved context
and cites its sources.

**Results.** Retrieval achieved a high `hit@k` on the evaluation set, in-corpus
questions were answered correctly with citations, and the system correctly
abstained on an out-of-corpus question — demonstrating the anti-hallucination
behaviour that motivated the project.

**Challenges.**
- *Chunking trade-offs*: chunks too large dilute relevance; too small lose
  context. The 1000/150 setting balanced both for this corpus.
- *Faithfulness vs. helpfulness*: a strict "answer only from context" prompt can
  make the model abstain too eagerly; the prompt was tuned to allow concise
  synthesis across retrieved chunks.
- *API key & cost*: every query incurs embedding + completion cost; persisting
  the FAISS index avoids re-embedding.

**Future improvements.**
- Add a **re-ranking** step (e.g. a cross-encoder) after vector retrieval.
- **Hybrid search**: combine BM25 keyword search with dense vectors.
- Stream responses and add **conversation memory** for follow-up questions.
- Automated faithfulness scoring with a framework such as **RAGAS**.
- Swap FAISS for a managed vector DB (Pinecone/pgvector) for scale & persistence.

The deployment of this pipeline as an interactive web app is in
[`app.py`](app.py) (Streamlit).
